In [ ]:
from ml_enhance import QuantumFPDatasetBuilder, RDKitFeatureCalculator, get_preprocessed_smiles, canonicalize_smiles
from pathlib import Path
from rdkit import Chem
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def build_solubility_lookup(raw_df: pd.DataFrame) -> pd.Series:
    """
    Returns a Series indexed by canonical SMILES containing solubility.
    """
    df = raw_df[["SMILES", "Solubility"]].copy()

    df["canon_smiles"] = df["SMILES"].apply(canonicalize_smiles)
    df = df.dropna(subset=["canon_smiles"])

    return df.set_index("canon_smiles")["Solubility"]

def attach_solubility(feature_df: pd.DataFrame, sol_lookup: pd.Series) -> pd.DataFrame:
    df = feature_df.copy()

    df["canon_smiles"] = df["smiles"].apply(canonicalize_smiles)
    df["solubility"] = df["canon_smiles"].map(sol_lookup)

    return df

In [ ]:
DATA_SOURCE = Path("../data/AqSolDB/data_curated.csv")
raw_data = pd.read_csv(DATA_SOURCE)
# QFP_preprocessed_smiles = np.array([get_preprocessed_smiles(smiles) for smiles in raw_data["SMILES"]])
# Write the smiles to JSON
# Run data through QuantumFP

In [ ]:
QFP_OUTPUT_PATH = Path("../data/QuantumFP/QFP_output")
builder = QuantumFPDatasetBuilder(QFP_OUTPUT_PATH)
df, errors = builder.build_dataset_batched(batch_size=100)

In [ ]:
errors

In [ ]:
rdkit_calculator = RDKitFeatureCalculator()
df = rdkit_calculator.add_to_dataframe(df, multiprocess=True)

In [ ]:
# Add solutbility:
solubility_lookup = build_solubility_lookup(raw_data)
df = attach_solubility(df, solubility_lookup)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df_clean = df.dropna() # mostly consisting of the molecules with transition metals as they cause problems with the partial charge calculation of RDKit
df_clean.info()

In [ ]:
df_clean.to_csv("../data/processed_dataset_wo_metals_w_more_qm.csv", index=False)

In [ ]:
from collections import Counter

df_clean = df.dropna()

print()

total_atom_types = []
count = 0
for smiles in df_clean["canon_smiles"]:
    mol = Chem.MolFromSmiles(smiles)

    mol_atom_types = [atom.GetSymbol() for atom in mol.GetAtoms()]

    total_atom_types.extend(mol_atom_types)

print(Counter(total_atom_types))

In [ ]:
Counter({'C': 104752, 'O': 22047, 'N': 10486, 'Cl': 3754, 'S': 1874, 'F': 1204, 'Br': 474, 'P': 332, 'I': 133, 'Si': 93, 'Sn': 18, 'Se': 10, 'B': 9, 'Pb': 6, 'Co': 5, 'Zn': 4, 'Cu': 4, 'As': 4, 'V': 3, 'Mo': 3, 'Cd': 3, 'Mn': 2, 'W': 2, 'Nb': 2, 'Cr': 2, 'Te': 2, 'Sb': 2, 'Ge': 2, 'Ca': 1, 'Bi': 1, 'Al': 1, 'Sr': 1, 'H': 1, 'Fe': 1, 'Ni': 1, 'Ti': 1, 'Zr': 1, 'Be': 1, 'Ce': 1, 'Hg': 1, 'Y': 1})